# Partie A : Data management et nettoyage (Lyon)

Données : Inside Airbnb, Lyon, snapshot du 17/09/2026 (licence CC BY 4.0). Le prof a ajouté des erreurs volontaires dans les fichiers.

Responsable : membre 1

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chemins : seule cellule à modifier
VILLE = "lyon"
DOSSIER = "../data/" + VILLE          # Colab : "/content/drive/MyDrive/data/" + VILLE
SORTIE = "../data_clean"
SEED = 42

print("pandas", pd.__version__)       # la signature change selon la version de pandas

Fonction signature de l'annexe A (copiée sans modification) :

In [ ]:
import hashlib
import pandas as pd


def signature(df, nom):
     """Résumé détaillé et empreinte d'une table nettoyée."""
     copie = df.copy()
     for col in copie.columns:
          # les listes (ex. amenities) sont converties en texte pour pouvoir être résumées
          if copie[col].map(lambda v: isinstance(v, (list, dict, set, tuple))).any():
              copie[col] = copie[col].map(str)
     empreinte = hashlib.md5(
          pd.util.hash_pandas_object(copie.astype(str), index=False).values.tobytes()
     ).hexdigest()


     print(f"=== Signature : {nom} ===")
     print(f"Lignes : {copie.shape[0]}          Colonnes : {copie.shape[1]}")
     print(f"Empreinte MD5 : {empreinte}")
     resume = pd.DataFrame({
          "type": copie.dtypes.astype(str),
          "manquants": copie.isna().sum(),
          "pct_manquants": (copie.isna().mean() * 100).round(2),
          "valeurs_distinctes": copie.nunique(),
     })
     display(resume)
     numeriques = copie.select_dtypes("number")
     if not numeriques.empty:
          display(numeriques.describe().T.round(2))
     return empreinte

In [ ]:
listings_brut = pd.read_csv(DOSSIER + "/listings.csv.gz", low_memory=False)
reviews_brut = pd.read_csv(DOSSIER + "/reviews.csv.gz")
quartiers = pd.read_csv(DOSSIER + "/neighbourhoods.csv")

print("listings :", listings_brut.shape)
print("reviews :", reviews_brut.shape)
print("quartiers :", quartiers.shape)

## A1 : Cadrage

### Les questions et les indicateurs

| Question | Indicateur | Colonnes |
|---|---|---|
| Q1 Taille du marché, saisonnalité | Nombre d'annonces actives (au moins 1 avis sur 12 mois et des jours disponibles) ; nombre d'avis par mois | `number_of_reviews_ltm`, `availability_365`, reviews `date` |
| Q2 Particuliers ou professionnels | Part des annonces d'hôtes qui ont 2 annonces ou plus ; part détenue par les 10 % d'hôtes les plus gros | `host_id`, `calculated_host_listings_count` |
| Q3 Meilleurs quartiers | Revenu médian, occupation médiane, annonces pour 1 000 habitants par arrondissement | `neighbourhood_cleansed`, `estimated_revenue_l365d`, `estimated_occupancy_l365d` |
| Q4 Droit de louer | Part des annonces avec un numéro d'enregistrement, en bail mobilité, sans numéro ; part des annonces avec 30 nuits minimum ou plus | `license`, `minimum_nights` |
| Q5 Le produit qui rapporte | Prix et revenu médians selon le type de logement, la capacité et les équipements | `room_type`, `accommodates`, `bedrooms`, `amenities`, `price` |
| Q6 (bonus) Ce que veulent les voyageurs | Mots les plus fréquents dans les avis, avis positifs ou négatifs | reviews `comments`, `review_scores_rating` |

On utilise des médianes plutôt que des moyennes, car les prix et les revenus ont quelques valeurs très élevées.

### La réglementation à Lyon

| | Règle | Sources |
|---|---|---|
| 1. Enregistrement | Obligatoire pour tout meublé de tourisme. Déclaration en ligne à la Ville de Lyon, le numéro (13 caractères) doit être affiché sur l'annonce. | [lyon.fr](https://www.lyon.fr/node/838135), [entreprises.gouv.fr](https://www.entreprises.gouv.fr/fr/mot-cle/location-touristique) |
| 2. Résidence principale | Location possible 90 nuits par an maximum depuis le 1er janvier 2026 (avant : 120). | [lyon.fr](https://www.lyon.fr/node/838135), [Lyon Capitale](https://www.lyoncapitale.fr/actualite/la-ville-de-lyon-adopte-trois-mesures-pour-faire-face-a-la-crise-du-logement) |
| 3. Résidence secondaire, investissement | Il faut une autorisation de changement d'usage. Dans l'hypercentre et pour les logements de 35 m² ou plus, il faut une compensation : transformer un autre local en logement, dans le même arrondissement. | [lyon.fr](https://www.lyon.fr/node/838135) |
| 4. Durée minimale | La loi ne donne pas de nombre de nuits ("courtes durées", "clientèle de passage"). On prend moins de 30 nuits, car le bail mobilité dure de 1 à 10 mois. | [service-public.fr](https://www.service-public.gouv.fr/particuliers/vosdroits/F34759), [CMS](https://cms.law/en/fra/news-information/location-de-meubles-touristiques-de-courte-duree-et-changement-d-usage) |
| 5. Sanctions | Changement d'usage sans autorisation : jusqu'à 100 000 € d'amende. Pas de déclaration : jusqu'à 10 000 €. Les règles se durcissent (loi Le Meur 2024 : 90 nuits, enregistrement national, DPE). | [notaires.fr](https://www.notaires.fr/fr/article/location-les-regles-applicables-aux-meubles-de-tourisme), [AGN Avocats](https://www.agn-avocats.fr/blog/immobilier/loi-le-meur-et-meubles-de-tourisme-enregistrement-mai-2026/) |

Pour un investisseur, le point bloquant est la ligne 3 : un logement acheté pour la location courte durée demande presque toujours une compensation.

### Données personnelles

- On supprime : le prénom, la présentation et la photo de l'hôte (`host_name`, `host_about`, `host_picture_url`, `host_url`...), le nom et la description de l'annonce, et le prénom et l'identifiant des voyageurs (`reviewer_name`, `reviewer_id`).
- On garde `host_id` pour compter les annonces par hôte (Q2), sans jamais l'afficher.
- On garde le texte des avis (`comments`) pour la partie D, mais on n'affiche jamais un avis entier, car il contient souvent des prénoms.

## A2 : Exploration et dictionnaire

In [ ]:
listings_brut.info()

In [ ]:
# Pourcentage de valeurs manquantes par colonne
manquants = (listings_brut.isna().mean() * 100).round(1)
print(manquants.sort_values(ascending=False).head(20))

In [ ]:
vides = manquants[manquants == 100]
print(len(vides), "colonnes complètement vides :")
print(list(vides.index))

In [ ]:
reviews_brut.info()
print("id d'avis en double :", reviews_brut["id"].duplicated().sum())
print(reviews_brut.nunique())

Remarques : 14 colonnes sont vides à Lyon (dont `bathrooms`, `host_response_rate`, `host_acceptance_rate`, `instant_bookable`). On les laisse de côté. Le nombre de salles de bain viendra de `bathrooms_text`. Dans les avis, des `id` sont en double : on les traite en A5.

### Dictionnaire des 26 colonnes retenues

| Colonne | Signification | Type cible | Unité | Décision |
|---|---|---|---|---|
| id | identifiant de l'annonce | entier | - | garder |
| host_id | identifiant de l'hôte | entier | - | garder (jamais affiché) |
| calculated_host_listings_count | nombre d'annonces de l'hôte à Lyon | entier | annonces | garder |
| host_is_superhost | hôte Superhôte ou non | booléen | - | transformer |
| neighbourhood_cleansed | arrondissement | texte | - | transformer |
| latitude | latitude | décimal | degrés | transformer |
| longitude | longitude | décimal | degrés | transformer |
| property_type | type de bien | texte | - | garder |
| room_type | logement entier, chambre... | texte | - | transformer |
| accommodates | capacité | entier | personnes | garder |
| bedrooms | chambres | décimal | chambres | garder |
| beds | lits | décimal | lits | garder |
| bathrooms_text | salles de bain (texte) | texte | - | transformer en nb_sdb et sdb_partagee |
| amenities | équipements | liste | - | transformer |
| price | prix par nuit | décimal | € | transformer |
| price_quote_price_per_night | prix d'un devis réel | décimal | € | écarter après A5 (sert de contrôle) |
| minimum_nights | durée minimale | décimal | nuits | garder |
| availability_365 | jours disponibles sur un an | entier | jours | garder |
| number_of_reviews | nombre d'avis | entier | avis | garder |
| number_of_reviews_ltm | avis des 12 derniers mois | entier | avis | garder |
| first_review | date du premier avis | date | - | transformer |
| last_review | date du dernier avis | date | - | transformer |
| review_scores_rating | note globale | décimal | sur 5 | garder |
| estimated_occupancy_l365d | nuits réservées estimées sur un an | entier | nuits | garder |
| estimated_revenue_l365d | revenu estimé sur un an | décimal | € | garder |
| license | numéro d'enregistrement | texte | - | garder |

Les trois dernières colonnes de prix, d'occupation et de revenu ne sont pas dans le dictionnaire officiel : leur sens vient de leur nom et de la page "Hypothèses de données" d'Inside Airbnb.

In [ ]:
colonnes = ["id", "host_id", "calculated_host_listings_count", "host_is_superhost",
            "neighbourhood_cleansed", "latitude", "longitude", "property_type", "room_type",
            "accommodates", "bedrooms", "beds", "bathrooms_text", "amenities", "price",
            "price_quote_price_per_night", "minimum_nights", "availability_365",
            "number_of_reviews", "number_of_reviews_ltm", "first_review", "last_review",
            "review_scores_rating", "estimated_occupancy_l365d", "estimated_revenue_l365d", "license"]

# Type d'origine, % de manquants et nombre de valeurs distinctes des colonnes retenues
dictionnaire = pd.DataFrame({"type_origine": listings_brut[colonnes].dtypes,
                             "pct_manquants": manquants[colonnes],
                             "valeurs_distinctes": listings_brut[colonnes].nunique()})
dictionnaire

## A3 : Sélection et typage

In [ ]:
listings = listings_brut[colonnes].copy()
reviews = reviews_brut[["id", "listing_id", "date", "comments"]].copy()
print(listings.shape, reviews.shape)

### Le prix

In [ ]:
# Les différentes écritures du prix (exemples)
print(listings["price"].dropna().sample(15, random_state=SEED).tolist())

In [ ]:
def convertir_prix(texte):
    if pd.isna(texte):
        return np.nan
    texte = texte.replace("$", "").replace("CAD", "").replace("dollars", "").replace(" ", "")
    if "," in texte and "." not in texte:
        texte = texte.replace(",", ".")     # 180,00 -> 180.00
    else:
        texte = texte.replace(",", "")      # 2,388.45 -> 2388.45
    return float(texte)

listings["price"] = listings["price"].map(convertir_prix)
illisibles = listings_brut["price"].notna().sum() - listings["price"].notna().sum()
print("prix illisibles :", illisibles)

La devise est l'euro : le $ vient de l'export d'Inside Airbnb (voir dictionnaire officiel) et les devis réels sont en EUR. Les prix écrits "CAD" ou "dollars" sont égaux au devis, ce sont juste des écritures différentes.

Les taux d'hôte (`host_response_rate`, `host_acceptance_rate`) sont vides à 100 % à Lyon : rien à convertir.

### Les booléens

In [ ]:
print(listings["host_is_superhost"].value_counts())

In [ ]:
vrai_faux = {"t": True, "TRUE": True, "Vrai": True, "yes": True, "1": True,
             "f": False, "FALSE": False, "Faux": False, "no": False, "0": False}
listings["host_is_superhost"] = listings["host_is_superhost"].map(vrai_faux)
print("illisibles :", listings["host_is_superhost"].isna().sum())

### Les dates

In [ ]:
# Deux formats : 2024-06-30 et 30/06/2024
def convertir_date(colonne):
    date1 = pd.to_datetime(colonne, format="%Y-%m-%d", errors="coerce")
    date2 = pd.to_datetime(colonne, format="%d/%m/%Y", errors="coerce")
    return date1.fillna(date2)

listings["first_review"] = convertir_date(listings["first_review"])
listings["last_review"] = convertir_date(listings["last_review"])
reviews["date"] = convertir_date(reviews["date"])

print("illisibles first_review :", listings_brut["first_review"].notna().sum() - listings["first_review"].notna().sum())
print("illisibles last_review :", listings_brut["last_review"].notna().sum() - listings["last_review"].notna().sum())
print("illisibles date des avis :", reviews_brut["date"].notna().sum() - reviews["date"].notna().sum())

### Salles de bain et équipements

In [ ]:
def nombre_sdb(texte):
    if pd.isna(texte):
        return np.nan
    if "half" in texte.lower():
        return 0.5                           # "Half-bath" = demi salle de bain
    return float(texte.split(" ")[0])        # "1.5 baths" -> 1.5

listings["nb_sdb"] = listings["bathrooms_text"].map(nombre_sdb)
listings["sdb_partagee"] = listings["bathrooms_text"].str.lower().str.contains("shared", na=False)   # "1 shared bath"
listings = listings.drop(columns="bathrooms_text")

# amenities est un texte qui ressemble à une liste : json.loads le transforme en vraie liste
listings["amenities"] = listings["amenities"].map(json.loads)
listings["nb_equipements"] = listings["amenities"].map(len)
print("salles de bain partagées :", listings["sdb_partagee"].sum())
print(listings[["nb_sdb", "nb_equipements"]].describe())

### Type de logement et quartier

In [ ]:
print(listings["room_type"].value_counts())

In [ ]:
types = {"entire home/apt": "Entire home/apt", "entire home": "Entire home/apt",
         "private room": "Private room", "private": "Private room",
         "shared room": "Shared room", "hotel room": "Hotel room"}
listings["room_type"] = listings["room_type"].str.strip().str.lower().map(types)
print(listings["room_type"].value_counts(dropna=False))

In [ ]:
print(listings["neighbourhood_cleansed"].nunique(), "écritures pour", len(quartiers), "arrondissements")

# On enlève les espaces et les majuscules, puis on remet le nom officiel
noms_officiels = {}
for nom in quartiers["neighbourhood"]:
    noms_officiels[nom.lower()] = nom
listings["neighbourhood_cleansed"] = listings["neighbourhood_cleansed"].str.strip().str.lower().map(noms_officiels)
print(listings["neighbourhood_cleansed"].value_counts(dropna=False))

## A4 : Valeurs manquantes

In [ ]:
print((listings.isna().mean() * 100).round(1).sort_values(ascending=False).head(12))

In [ ]:
# Pourquoi le prix manque-t-il ? On regarde si l'annonce est réservable
pd.crosstab(listings["price"].isna(), listings["availability_365"] == 0,
            rownames=["prix manquant"], colnames=["0 jour disponible"])

In [ ]:
# Pourquoi la note manque-t-elle ? On regarde si l'annonce a des avis
pd.crosstab(listings["review_scores_rating"].isna(), listings["number_of_reviews"] == 0,
            rownames=["note manquante"], colnames=["0 avis"])

| Colonne | % manquant | Stratégie | Pourquoi |
|---|---|---|---|
| price, estimated_revenue_l365d | 45 % | on laisse vide + colonne `prix_manquant` | la plupart des annonces sans prix ne sont pas réservables (0 jour disponible) : inventer un prix inventerait un revenu |
| first_review, last_review, review_scores_rating | 18 % | on laisse vide | ces annonces n'ont aucun avis, il n'y a donc pas de note |
| license | 37 % | catégorie "Non renseigné" | l'absence de numéro est une information pour Q4 |
| beds, bedrooms | 1,5 % et moins | médiane par capacité | le nombre de lits dépend de la capacité |
| nb_sdb | 0,1 % | médiane | très peu de lignes |
| minimum_nights | 0,7 % | mode (valeur la plus fréquente) | très peu de lignes |
| comments (avis) | 0,3 % | on garde l'avis sans texte | il compte quand même pour la saisonnalité (Q1) |

In [ ]:
listings["prix_manquant"] = listings["price"].isna()
listings["license"] = listings["license"].fillna("Non renseigné")

listings["beds"] = listings["beds"].fillna(listings.groupby("accommodates")["beds"].transform("median"))
listings["bedrooms"] = listings["bedrooms"].fillna(listings.groupby("accommodates")["bedrooms"].transform("median"))
listings["nb_sdb"] = listings["nb_sdb"].fillna(listings["nb_sdb"].median())
listings["minimum_nights"] = listings["minimum_nights"].fillna(listings["minimum_nights"].mode()[0])

print(listings.isna().sum()[listings.isna().sum() > 0])

## A5 : Doublons, incohérences et valeurs aberrantes

### Doublons dans listings

In [ ]:
# Doublons exacts (on ne compare pas amenities, qui contient des listes)
sans_liste = listings.columns.drop("amenities")
doublons_exacts = listings.duplicated(subset=sans_liste)
print("doublons exacts :", doublons_exacts.sum())

rejets_listings = listings[doublons_exacts].copy()
rejets_listings["motif"] = "doublon exact"
listings = listings[~doublons_exacts].copy()

In [ ]:
# Doublons partiels : même id, mais des valeurs différentes
meme_id = listings["id"].duplicated(keep=False)
print("lignes avec un id en double :", meme_id.sum())
listings[meme_id].sort_values("id")[["id", "price", "price_quote_price_per_night"]].head(6)

Seul le prix change. Une version est égale au prix du devis réel, l'autre est 10 % plus chère. On garde la version égale au devis.

In [ ]:
a_ecarter = meme_id & (listings["price"] != listings["price_quote_price_per_night"])
print("doublons partiels écartés :", a_ecarter.sum())

rejet = listings[a_ecarter].copy()
rejet["motif"] = "doublon partiel"
rejets_listings = pd.concat([rejets_listings, rejet])
listings = listings[~a_ecarter].copy()
print("id encore en double :", listings["id"].duplicated().sum())

### Doublons dans reviews

In [ ]:
doublons_exacts = reviews.duplicated()
print("doublons exacts :", doublons_exacts.sum())
rejets_reviews = reviews[doublons_exacts].copy()
rejets_reviews["motif"] = "doublon exact"
reviews = reviews[~doublons_exacts].copy()

# Doublons partiels : le même avis (même id) rattaché à plusieurs annonces. On garde la première ligne.
doublons_partiels = reviews["id"].duplicated()
print("doublons partiels :", doublons_partiels.sum())
rejet = reviews[doublons_partiels].copy()
rejet["motif"] = "doublon partiel"
rejets_reviews = pd.concat([rejets_reviews, rejet])
reviews = reviews[~doublons_partiels].copy()

### Valeurs impossibles dans listings

In [ ]:
# Coordonnées : Lyon est entre 45.70 et 45.82 de latitude, 4.76 et 4.91 de longitude
inversees = listings["latitude"].between(4.76, 4.91) & listings["longitude"].between(45.70, 45.82)
zero = (listings["latitude"] == 0) & (listings["longitude"] == 0)
print("coordonnées inversées :", inversees.sum(), "| coordonnées à 0 :", zero.sum())

# Inversées : on échange les deux colonnes. A zéro : position inconnue, on met vide
ancienne_latitude = listings.loc[inversees, "latitude"]
listings.loc[inversees, "latitude"] = listings.loc[inversees, "longitude"]
listings.loc[inversees, "longitude"] = ancienne_latitude
listings.loc[zero, ["latitude", "longitude"]] = np.nan

In [ ]:
# Capacité : 0 ou 99 personnes est impossible (maximum Airbnb : 16)
print(listings["accommodates"].value_counts().sort_index())
capacite_impossible = (listings["accommodates"] == 0) | (listings["accommodates"] > 16)
print("capacités impossibles :", capacite_impossible.sum())

rejet = listings[capacite_impossible].copy()
rejet["motif"] = "capacité impossible"
rejets_listings = pd.concat([rejets_listings, rejet])
listings = listings[~capacite_impossible].copy()

In [ ]:
# Durée minimale négative : c'est une erreur de signe, on prend la valeur positive
negatives = listings["minimum_nights"] < 0
print("durées minimales négatives :", negatives.sum())
listings.loc[negatives, "minimum_nights"] = listings.loc[negatives, "minimum_nights"].abs()

In [ ]:
# Prix impossibles : négatif, 0, ou 99999. On le remplace par le prix du devis réel de l'annonce
prix_impossible = (listings["price"] <= 0) | (listings["price"] == 99999)
print("prix impossibles :", prix_impossible.sum())
listings.loc[prix_impossible, "price"] = listings.loc[prix_impossible, "price_quote_price_per_night"]

In [ ]:
# Note hors de l'échelle 1 à 5 (-1 ou 99) : la vraie note est inconnue, on met vide
note_impossible = listings["review_scores_rating"].notna() & ~listings["review_scores_rating"].between(1, 5)
print("notes impossibles :", note_impossible.sum())
listings.loc[note_impossible, "review_scores_rating"] = np.nan

In [ ]:
# Dates : dernier avis après la collecte (19/09/2026) ou premier avis après le dernier
date_collecte = pd.to_datetime("2026-09-19")
dates_impossibles = (listings["last_review"] > date_collecte) | (listings["first_review"] > listings["last_review"])
print("dates impossibles :", dates_impossibles.sum())
listings.loc[dates_impossibles, ["first_review", "last_review"]] = pd.NaT

### Valeurs impossibles dans reviews

In [ ]:
# Avis daté après la collecte
avis_futurs = reviews["date"] > date_collecte
# Avis d'une annonce qui n'existe pas (ou qui a été écartée)
avis_orphelins = ~reviews["listing_id"].isin(listings["id"])
print("avis futurs :", avis_futurs.sum(), "| avis orphelins :", avis_orphelins.sum())

rejet = reviews[avis_futurs].copy()
rejet["motif"] = "date après la collecte"
rejets_reviews = pd.concat([rejets_reviews, rejet])
reviews = reviews[~avis_futurs].copy()

avis_orphelins = ~reviews["listing_id"].isin(listings["id"])
rejet = reviews[avis_orphelins].copy()
rejet["motif"] = "annonce absente"
rejets_reviews = pd.concat([rejets_reviews, rejet])
reviews = reviews[~avis_orphelins].copy()

### Valeurs aberrantes : prix et durée minimale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
sns.boxplot(x=listings["price"], ax=axes[0])
axes[0].set_title("Prix par nuit (€)")
sns.boxplot(x=listings["minimum_nights"], ax=axes[1])
axes[1].set_title("Durée minimale (nuits)")
plt.show()

Q1 = listings["price"].quantile(0.25)
Q3 = listings["price"].quantile(0.75)
IQR = Q3 - Q1
seuil = Q3 + 3 * IQR
print("Q1 =", Q1, "| Q3 =", Q3, "| seuil =", seuil)

In [ ]:
# Les prix les plus élevés : ce sont de grands logements
listings.sort_values("price", ascending=False)[["price", "accommodates", "bedrooms"]].head(8)

Choix pour le prix : on garde les prix au-dessus du seuil, car ce sont de grands logements (10 à 16 personnes). On les signale avec la colonne `prix_extreme` pour pouvoir les exclure des moyennes en B et C.

Choix pour la durée minimale : 30 nuits (bail mobilité) et 365 nuits (annonce fermée à la courte durée) sont fréquentes et utiles pour Q4, on les garde. Au-delà de 365 nuits, on ramène à 365.

In [ ]:
listings["prix_extreme"] = listings["price"] > seuil
print("prix extrêmes :", listings["prix_extreme"].sum())

trop_long = listings["minimum_nights"] > 365
print("durées supérieures à 365 :", trop_long.sum())
listings.loc[trop_long, "minimum_nights"] = 365

### Tableau des anomalies et rejets

| Fichier | Anomalie | Colonnes | Lignes | Traitement |
|---|---|---|---|---|
| listings | doublons exacts | toutes | 94 | écartés |
| listings | doublons partiels (prix +10 %) | id, price | 26 | on garde la version égale au devis |
| listings | coordonnées inversées | latitude, longitude | 28 | remises dans le bon ordre |
| listings | coordonnées à 0 | latitude, longitude | 9 | mises à vide |
| listings | capacité 0 ou 99 | accommodates | 28 | écartées |
| listings | durée minimale négative | minimum_nights | 18 | signe corrigé |
| listings | prix négatif, 0 ou 99999 | price | 55 | remplacé par le devis réel |
| listings | note -1 ou 99 | review_scores_rating | 28 | mise à vide |
| listings | dates d'avis impossibles | first_review, last_review | 64 | mises à vide |
| listings | prix très élevés | price | 155 | gardés, signalés (prix_extreme) |
| listings | durée minimale > 365 | minimum_nights | 3 | ramenée à 365 |
| reviews | doublons exacts | toutes | 4 733 | écartés |
| reviews | doublons partiels (même id) | id | 2 235 | on garde le premier |
| reviews | avis en 2027 | date | 469 | écartés |
| reviews | avis d'une annonce absente | listing_id | 2 763 | écartés |

In [ ]:
print("Annonces écartées :")
print(rejets_listings["motif"].value_counts())
print("\nAvis écartés :")
print(rejets_reviews["motif"].value_counts())

## A6 : Contrôles, bilan et signature

In [ ]:
# La colonne du devis a servi de contrôle, on la retire
listings = listings.drop(columns="price_quote_price_per_night")

# On trie par id pour que la signature soit toujours la même
listings = listings.sort_values("id").reset_index(drop=True)
reviews = reviews.sort_values("id").reset_index(drop=True)

In [ ]:
# Chaque contrôle compte les lignes en erreur : on attend 0 partout
print("1. id en double :", listings["id"].duplicated().sum())
print("2. prix <= 0 :", (listings["price"] <= 0).sum())
print("3. coordonnées hors de Lyon :", (listings["latitude"].notna() & ~listings["latitude"].between(45.70, 45.82)).sum())
print("4. premier avis après le dernier :", (listings["first_review"] > listings["last_review"]).sum())
print("5. dernier avis après la collecte :", (listings["last_review"] > date_collecte).sum())
print("6. quartier inconnu :", (~listings["neighbourhood_cleansed"].isin(quartiers["neighbourhood"])).sum())
print("7. availability_365 hors de 0 à 365 :", (~listings["availability_365"].between(0, 365)).sum())
print("8. capacité hors de 1 à 16 :", (~listings["accommodates"].between(1, 16)).sum())
print("9. durée minimale hors de 1 à 365 :", (~listings["minimum_nights"].between(1, 365)).sum())
print("10. note hors de 1 à 5 :", (listings["review_scores_rating"].notna() & ~listings["review_scores_rating"].between(1, 5)).sum())
print("11. type de logement manquant :", listings["room_type"].isna().sum())
print("12. avis en double :", reviews["id"].duplicated().sum())
print("13. avis après la collecte :", (reviews["date"] > date_collecte).sum())
print("14. avis sans annonce :", (~reviews["listing_id"].isin(listings["id"])).sum())

In [ ]:
bilan = pd.DataFrame({
    "listings avant": [listings_brut.shape[0], listings_brut.shape[1], round(listings_brut.isna().mean().mean() * 100, 1)],
    "listings après": [listings.shape[0], listings.shape[1], round(listings.isna().mean().mean() * 100, 1)],
    "reviews avant": [reviews_brut.shape[0], reviews_brut.shape[1], round(reviews_brut.isna().mean().mean() * 100, 1)],
    "reviews après": [reviews.shape[0], reviews.shape[1], round(reviews.isna().mean().mean() * 100, 1)],
}, index=["lignes", "colonnes", "% manquants"], dtype=object)
bilan.loc["lignes rejetées"] = ["-", len(rejets_listings), "-", len(rejets_reviews)]
bilan

In [ ]:
signature_listings = signature(listings, "listings nettoyé")

In [ ]:
signature_reviews = signature(reviews, "reviews nettoyé")

La signature change selon la version de pandas : toute l'équipe utilise pandas 2.3.3 (requirements.txt).

In [ ]:
# Format parquet : garde les types (dates, listes d'équipements), contrairement au csv
listings.to_parquet(SORTIE + "/listings_propre.parquet", index=False)
reviews.to_parquet(SORTIE + "/reviews_propre.parquet", index=False)
print("tables enregistrées dans", SORTIE)